# Aula 3 · Lab 1 — Tópicos, partições e réplicas

## Objetivo
Entender como o Kafka organiza os dados fisicamente: um **tópico** é dividido em **partições**, e cada
partição é copiada em **réplicas** espalhadas pelos brokers. Ao final do lab você deve saber explicar:

- por que partições são a unidade de **paralelismo** e de **ordenação** do Kafka;
- por que réplicas são a unidade de **tolerância a falhas**;
- o que são **líder**, **seguidor** e **ISR**;
- como ler essas informações por código (Python) e pela linha de comando.

## O ambiente
Nosso cluster tem 3 brokers (`kafka1`, `kafka2`, `kafka3`) rodando em modo **KRaft**. Nesse modo, os
próprios brokers guardam os metadados do cluster (quais tópicos existem, quem lidera cada partição),
sem depender do ZooKeeper, que era obrigatório até o Kafka 3.x e foi removido no Kafka 4.0.

Cada broker aqui acumula duas funções:
- **broker**: armazena partições e atende produtores e consumidores;
- **controller**: participa do quórum que decide os metadados. Um dos três é o *controller ativo* (o "líder" do quórum).

> Abra a **Kafka UI** (porta 8080) numa aba ao lado. Tudo o que fizermos aqui aparece lá.

## Etapa 1 — Conectando e lendo os metadados do cluster

O `AdminClient` é o cliente administrativo do Kafka: cria e apaga tópicos, lê configurações e
descreve o cluster. Ele recebe o parâmetro **`bootstrap.servers`**, que é só o *ponto de entrada*:
o cliente se conecta a qualquer um desses endereços, pergunta "quem faz parte do cluster?" e, a partir daí,
conversa diretamente com cada broker. Por isso basta listar alguns brokers, não todos.

`list_topics()` devolve os **metadados** do cluster. Na saída, observe:

| Campo | O que significa |
|---|---|
| `cluster_id` | Identificador único do cluster (definido na formatação dos discos, igual nos 3 brokers) |
| `controller_id` | Qual broker é o controller ativo do quórum KRaft agora |
| `broker id -> host:porta` | Cada broker, seu id numérico (`KAFKA_NODE_ID`) e o endereço que ele anuncia aos clientes |

In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic, ConfigResource
from comum.config import BOOTSTRAP

admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
meta = admin.list_topics(timeout=10)

print("Cluster id:", meta.cluster_id)
print("Controller ativo:", meta.controller_id)
for b in sorted(meta.brokers.values(), key=lambda b: b.id):
    print(f"  broker {b.id} -> {b.host}:{b.port}")

**Resultado esperado:** 3 brokers (ids 1, 2 e 3), todos na porta 9092, e um deles como controller ativo.
Se aparecerem menos de 3, algum container está parado.

## Etapa 2 — Criando um tópico

Vamos criar o tópico `aula3-pedidos`. Os três parâmetros de `NewTopic` são as decisões mais
importantes no desenho de um tópico:

| Parâmetro | Valor | O que controla |
|---|---|---|
| `num_partitions` | 6 | Em quantos "pedaços" independentes o tópico é dividido. Define o **paralelismo máximo**: um grupo de consumidores consegue ter no máximo 6 membros trabalhando ao mesmo tempo. A ordem das mensagens só é garantida **dentro** de cada partição. |
| `replication_factor` | 3 | Quantas cópias de cada partição existem, cada uma num broker diferente. Com 3 cópias, o tópico sobrevive à perda de até 2 brokers sem perder dados. Não pode ser maior que o número de brokers. |
| `retention.ms` | 86 400 000 (24 h) | Por quanto tempo as mensagens ficam guardadas. Diferente de uma fila tradicional, o Kafka **não apaga a mensagem quando ela é lida**: ela fica disponível até vencer a retenção, e qualquer consumidor pode relê-la. |

`create_topics()` é assíncrono: devolve um "futuro" por tópico, e `f.result()` espera a confirmação.
Se o tópico já existir (por exemplo, ao rodar a célula de novo), aparece um aviso `TOPIC_ALREADY_EXISTS`, que pode ser ignorado.

In [ ]:
TOPICO = "aula3-pedidos"

futuros = admin.create_topics([NewTopic(TOPICO, num_partitions=6, replication_factor=3,
                                        config={"retention.ms": str(24 * 3600 * 1000)})])
for nome, f in futuros.items():
    try:
        f.result()
        print("criado:", nome)
    except Exception as e:
        print("aviso:", e)

## Etapa 3 — Quem é líder de cada partição?

Cada partição tem **uma réplica líder** e as demais são **seguidoras**:

- todas as **escritas** vão para o líder; os seguidores copiam do líder continuamente;
- as **leituras** também vão, por padrão, para o líder;
- se o líder cair, um seguidor que esteja em dia é promovido (veremos isso no Lab 4).

A função `descrever()` mostra, para cada partição:

| Campo | O que significa |
|---|---|
| `líder` | id do broker que é o líder da partição agora |
| `réplicas` | lista de brokers que guardam uma cópia. O **primeiro** da lista é o *líder preferido*: o Kafka tenta devolver a liderança a ele quando possível |
| `ISR` | *In-Sync Replicas*: réplicas que estão **em dia** com o líder. Um seguidor sai do ISR se ficar mais de `replica.lag.time.max.ms` (30 s por padrão) sem acompanhar o líder. Só réplicas do ISR podem virar líder sem perda de dados |

In [ ]:
def descrever(topico):
    t = admin.list_topics(topic=topico, timeout=10).topics[topico]
    print(f"{topico}: {len(t.partitions)} partições")
    for p in sorted(t.partitions.values(), key=lambda p: p.id):
        print(f"  partição {p.id}: líder={p.leader} réplicas={p.replicas} ISR={p.isrs}")

descrever(TOPICO)

**O que observar:**
- os líderes ficam **distribuídos** entre os brokers 1, 2 e 3 (cada um lidera ~2 das 6 partições). Isso
  espalha a carga de escrita: nenhum broker recebe todo o tráfego do tópico;
- com o cluster saudável, `ISR` é igual a `réplicas` (3 brokers em cada);
- a ordem da lista de réplicas varia entre partições: é assim que o Kafka distribui os líderes preferidos.

Ligação com a teoria: **liderança e ISR são por partição, não por broker**. Um mesmo broker é líder de
algumas partições e seguidor de outras ao mesmo tempo.

### A mesma informação pela linha de comando

O Kafka vem com scripts de administração (`kafka-*.sh`). Abra um **terminal** no Jupyter
(*File → New → Terminal*) e rode:

```bash
kafka-topics.sh --bootstrap-server kafka1:9092 --list
kafka-topics.sh --bootstrap-server kafka1:9092 --describe --topic aula3-pedidos
kafka-configs.sh --bootstrap-server kafka1:9092 --entity-type topics --entity-name aula3-pedidos --describe
```

- `--list`: nomes de todos os tópicos (os que começam com `__`, como `__consumer_offsets`, são internos do Kafka);
- `--describe`: a primeira linha traz `PartitionCount`, `ReplicationFactor` e as configurações alteradas; as seguintes
  mostram `Leader`, `Replicas` e `Isr` de cada partição, os mesmos campos da Etapa 3;
- `kafka-configs.sh --describe`: só as configurações que foram alteradas em relação ao padrão (aqui, `retention.ms`).

## Etapa 4 — Configurações de um tópico

Cada tópico tem dezenas de configurações. Quando não são definidas na criação, valem os padrões do broker.
`describe_configs()` lê todas; vamos olhar as quatro mais importantes:

| Configuração | O que significa |
|---|---|
| `retention.ms` | Tempo de retenção das mensagens (definimos 24 h). `-1` significa guardar para sempre |
| `cleanup.policy` | O que fazer com dados antigos. `delete`: apaga segmentos vencidos pela retenção. `compact`: guarda só a **última** mensagem de cada chave, o que transforma o tópico numa espécie de "tabela" do estado atual |
| `min.insync.replicas` | Quantas réplicas do ISR precisam confirmar uma escrita feita com `acks=all`. Configuramos o cluster com **2**: com 3 réplicas, o tópico aceita escritas mesmo com 1 broker fora, mas recusa se sobrar só 1 (Lab 4) |
| `segment.bytes` | Tamanho de cada arquivo de log em disco (1 GB por padrão). A retenção apaga **segmentos inteiros**, não mensagens individuais; por isso uma mensagem pode durar um pouco mais que o `retention.ms` |

In [ ]:
recurso = ConfigResource(ConfigResource.Type.TOPIC, TOPICO)
configs = admin.describe_configs([recurso])[recurso].result()
for chave in ["retention.ms", "cleanup.policy", "min.insync.replicas", "segment.bytes"]:
    print(f"{chave:22s} = {configs[chave].value}")

**Resultado esperado:** `retention.ms = 86400000`, `cleanup.policy = delete`, `min.insync.replicas = 2`
e `segment.bytes = 1073741824`.

## Etapa 5 — Tópicos do curso

Os próximos labs (e as Aulas 4 e 5) usam os tópicos do **gerador de eventos** de uma loja fictícia:

| Tópico | Conteúdo | Chave da mensagem |
|---|---|---|
| `cliques` | usuário viu uma página ou produto | `usuario_id` |
| `pedidos` | usuário comprou um produto | `usuario_id` |
| `pagamentos` | resultado do pagamento, que chega segundos depois do pedido | `pedido_id` |
| `logins` | tentativas de login, incluindo "ataques" de força bruta simulados | `usuario_id` |
| `alertas` | saída dos detectores da Aula 5 | — |

`criar_topicos()` (em `comum/config.py`) cria todos com 3 partições e replicação 3, ignorando os que já existem.
A escolha da chave é importante: mensagens com a mesma chave vão para a mesma partição e, portanto,
ficam em ordem. Usar `usuario_id` garante que todos os eventos de um usuário sejam processados na sequência em que aconteceram.

In [ ]:
from comum.config import criar_topicos
criar_topicos()

## Exercícios
1. Crie um tópico `aula3-teste` com **1 partição** e replicação 3. Pela Kafka UI, descubra quem é o líder.
   Que limitação de paralelismo esse tópico tem?
2. Tente criar um tópico com `replication_factor=4`. Que erro aparece? Por quê?
3. Aumente o número de partições de `aula3-teste` para 4 (`kafka-topics.sh --alter --topic aula3-teste --partitions 4`).
   Dá para **diminuir** depois? Que problema o aumento causa para mensagens com chave?
4. Crie um tópico compactado (`cleanup.policy=compact`). Em que caso de uso ele faz sentido?